# Scikit-learn

Main features of `scikit-learn`:

- **Preprocessing**: data split, scaling, encoding, missing value imputation, ...
- **Supervised learning**: regression, classification, ...
- **Unsupervised learning**: clustering, dimension reduction, ...
- **Model evaluation**: measuring model performance based on some metrics.
- **Model selection**: choosing the best model or set of hyperparameters based on performance on validation data.
- **Pipeline and compose** (will not be covered)

The fundamental object in `scikit-learn` is `BaseEstimator`, and the heierarchy of the objects are as follows:
```text
BaseEstimator
├── TransformerMixin      ← Data Transformation (preprocessing, dimension reduction, ...)
├── ClassifierMixin       ← Estimator (classification)
├── RegressorMixin        ← Estimator (regression)
├── ClusterMixin, etc.    ← Estimator (clustering)
└── ...
```

## I. Data Preprocessing

### Data Split
- One of the main objectives of data science is to find a model that performs well on unseen (or new) data. 
- To do this, we split the dataset into a training set (for fitting the model), validation set (for model selection) and a test set (for final evaluation).
- After the split, you should **not** use any information of the test set.
- Before splitting the data, we assume that some data cleaning procedures that do not depend on the training data (any examples?) have already been applied.

In `scikit-learn`, the `train_test_split` function from the `sklearn.model_selection` module is used to randomly split the data into training and test sets based on the `test_size` parameter. Setting the `random_state` ensures that the split is reproducible.


```python
# Code template for data split
from sklearn.model_selection import train_test_split

TEST_PROPORTION = 0.2 # any float between 0 and 1
SEED = 100 # any integer

# Supervised learning
X = DataFrame_of_predictors
y = DataFrame_of_response_or_class_label # Ignore if unsupervised
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=TEST_PROPORTION, random_state=SEED)

# Unsupervised learning
X = DataFrame_of_predictors
X_train, X_test = train_test_split(X, test_size=TEST_PROPORTION, random_state=SEED)
```


### Transformation
- After splitting the data into training and test sets, we apply preprocessing steps such as:
  -  imputing missing values
  -  scaling numerical features
  -  encoding categorical variables.
- It is important that all transformations that learn from the data — like calculating means or encoding mappings — are fit only on the training set. (WHY?)
- Once fit, these transformers should then be applied to both the training and test data.
- We use `TransformerMixin` object for preprocessing.

**Examples**

| Transformer           | Description                                                 | Typical Use Case                       |
| --------------------- | ----------------------------------------------------------- | -------------------------------------- |
| `SimpleImputer`       | Fills in missing values with mean, median, or most frequent | Handling missing data                  |
| `StandardScaler`      | Standardizes features (zero mean, unit variance)            | Scaling numeric features               |
| `MinMaxScaler`        | Scales features to a given range (default: \[0, 1])         | Normalizing data for bounded models    |
| `RobustScaler`        | Scales using median and IQR (robust to outliers)            | Scaling with outliers                  |
| `OneHotEncoder`       | Converts categorical variables into binary columns          | Encoding nominal categorical features  |
| `OrdinalEncoder`      | Encodes categories as ordered integers                      | Encoding ordinal categorical features  |


Suppose you want to transform your dataframe `df` with the specific `Transformer`.

```python
# Code template for TransformerMixin object
df = DATA_TO_FIT_TRANSFORM
tf = Transformer(...)               # Instantiation
tf.fit(df)                          # Fitting
df_transformed = tf.transform(df)   # Transforming
```
---

Fitting and transformin can be done simultaneously by `fit_transform()`
```python
# Code template for TransformerMixin object (fit + transform)
tf = Transformer(...)                   # Instantiation
df_transformed = tf.fit_transform(df)   # Fitting + transforming
```
The object `tf` automaically saves the fit.

---

```python
## Which one is wrong?
X = pd.DataFrame(...)
X_train, X_test = train_test_split(X, test_size = 0.2, random_state = 87)

### (1) ###
tf = Transformer(...)
tf.fit(X_train)
X_train_transformed = tf.transform(X_train)
X_test_transformed = tf.transform(X_test)

### (2) ###
tf = Transformer(...)
tf.fit(X)
X_train_transformed = tf.transform(X_train)
X_test_transformed = tf.transform(X_test)
```

#### Exmaples

a) Missing value imputation

In [1]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy = 'mean') # strategy: mean, most_frequent, median, constant, ...

b) Scaling (Numerical variable)

When a model depends on distance calculations or involves gradient-based optimization, feature scaling becomes essential. Examples include:
| **Model**                        | **Reason**                                              |
| -------------------------------- | ---------------------------------------------------------------------- |
| `LogisticRegression`             | Optimization assumes features are on a similar scale                   |
| `Ridge`, `Lasso`, `ElasticNet`   | Regularization penalties are sensitive to feature magnitude            |
| `SVM           `                 | Distance-based; feature scales affect margin and kernel behavior       |
| `KNeighborsClassifier/Regressor` | Distance-based; dominant features distort results                      |
| `KMeans`                         | Uses Euclidean distance; scale mismatch skews clustering               |
| `PCA`                            | Based on variance; larger-scale features dominate principal components |


In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

scaler1 = StandardScaler()
scaler2 = MinMaxScaler()

d) Encoding (Categorical variable)

Encoding for categorical variables is necessary for any model that requires numerical input.

In [ ]:
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

encoder1 = OneHotEncoder()
encoder2 = OrdinalEncoder()

c) Dimensionality Reduction (Numerical variable)

- Even though dimensionality reduction techniques are unsupervised learning models, some of them are often applied during preprocessing to reduce noise, remove redundancy, or simplify the feature space. - Just like other data-dependent transformations, dimensionality reduction should be fit only on the training data and then applied to the test data.
- Due to their native mechanism, some reduction methods cannot be used for data preprocessing. An easy way to check this is whether they support **out-of-sample transform**. In `scikit-learn`, if a specific algorithm has the `transform()` method, it means it's okay to use for data preprocessing.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.manifold import Isomap

pca1 = PCA(n_components=2) # n_components: number of components to keep
pca2 = PCA(n_components=0.95) # if n_components is a float between 0 and 1, it means keep that percentage of variance
isomap = Isomap(n_neighbors=5, n_components=2) # n_neighbors: number of neighbors to consider, n_components: number of components to keep

#### 1.5 Save

After preprocessing, save 
- `X_train_processed`, `X_test_processed`, `y_train`, `y_test` in `data/processed` by using `to_csv()`.
- all the transformers you used (e.g. `imputer`, `scaler`, `encoder`) in `model` by using `pickle`.

In case you want to use multiple models and they require different data preprocessing, you should save them sepeartely. For each model, save:
- `X_train_processed`, `X_test_processed`, `y_train`, `y_test` in `data/processed/ModelName` by using `to_csv()`
- all the transformers you used (`num_imputer`, `cat_imputer`, `scaler`, `encoder`) in `model/ModelName` by using `pickle`.

#### IMPORTANT NOTE
- In addition to scikit-learn’s transformers, many preprocessing steps are often performed manually using pandas or custom logic.
- These include creating new features, grouping categories, handling outliers, or applying domain-specific transformations.
- You can perform such steps at any point in your workflow, as long as you ensure they are applied consistently to both training and test sets.
- Furthermore, as you conduct exploratory data analysis (EDA), you may identify patterns, anomalies, or relationships that reveal the need for additional preprocessing steps.